In [3]:
import os
import sys

# CRITICAL: Set environment variables BEFORE importing PySpark
# This must be done first, before any PySpark imports
os.environ['SPARK_HOME'] = '/opt/spark'
os.environ['JAVA_HOME'] = '/opt/java/openjdk'
os.environ['PYTHONPATH'] = '/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip:' + os.environ.get('PYTHONPATH', '')

# Add to Python path
sys.path.insert(0, '/opt/spark/python')
sys.path.insert(0, '/opt/spark/python/lib/py4j-0.10.9.7-src.zip')

# Verify environment is set correctly
print(f"SPARK_HOME: {repr(os.environ.get('SPARK_HOME'))}")
print(f"JAVA_HOME: {repr(os.environ.get('JAVA_HOME'))}")

# Verify JARs path exists
jars_path = '/opt/spark/assembly/target/scala-2.12/jars'
if os.path.exists(jars_path):
    print(f"✓ Spark JARs encontrados em: {jars_path}")
else:
    print(f"✗ ERRO: Spark JARs não encontrados em: {jars_path}")

# Now import PySpark
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder\
    .appName('init_env') \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

# Print Spark UI URL
print(f"\n✓ Spark iniciado com sucesso!")
print(f"✓ Spark UI disponível em: http://localhost:4040")
print(f"✓ Spark version: {spark.version}")

SPARK_HOME: '/opt/spark'
JAVA_HOME: '/opt/java/openjdk'
✓ Spark JARs encontrados em: /opt/spark/assembly/target/scala-2.12/jars

✓ Spark iniciado com sucesso!
✓ Spark UI disponível em: http://localhost:4040
✓ Spark version: 3.5.1


In [4]:
# Criar Schemas
schema_list = ['bronze','silver','gold']

for schema in schema_list:
    spark.sql(f"""
        CREATE DATABASE IF NOT EXISTS iceberg.{schema} COMMENT '' LOCATION 's3a://datalake/iceberg/{schema}/'
    """)

In [5]:
### Listar catalogs
spark.sql("SHOW databases").show()

+---------+
|namespace|
+---------+
|   bronze|
|  default|
|     gold|
|   silver|
+---------+



In [7]:
### Criar tabela teste
query = f"""CREATE TABLE IF NOT EXISTS iceberg.bronze.nyc_taxis
(
  vendor_id bigint,
  trip_id bigint,
  trip_distance float,
  fare_amount double,
  store_and_fwd_flag string
)
PARTITIONED BY (vendor_id);"""
spark.sql(query)

DataFrame[]

In [8]:
### Listar tablels em um schema
spark.sql("SHOW TABLES IN iceberg.bronze").show()


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   bronze|nyc_taxis|      false|
+---------+---------+-----------+



In [11]:
## Inserir valores
query = f"""INSERT INTO iceberg.bronze.nyc_taxis
VALUES (2, 1000371, 1.8, 15.32, 'Y'), (3, 1000372, 2.5, 22.15, 'N'), (5, 1000373, 0.9, 9.01, 'N'), (6, 1000374, 8.4, 42.13, 'Y');"""

display(spark.sql(query))

DataFrame[]

In [13]:
## Consultar dado
spark.sql("SELECT * FROM iceberg.bronze.nyc_taxis where store_and_fwd_flag ='Y' ").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        6|1000374|          8.4|      42.13|                 Y|
|        6|1000374|          8.4|      42.13|                 Y|
|        2|1000371|          1.8|      15.32|                 Y|
|        2|1000371|          1.8|      15.32|                 Y|
|        6|1000374|          8.4|      42.13|                 Y|
|        2|1000371|          1.8|      15.32|                 Y|
+---------+-------+-------------+-----------+------------------+



In [14]:
partitions = spark.sql("SELECT * FROM iceberg.bronze.nyc_taxis.history").show()


+--------------------+-------------------+-------------------+-------------------+
|     made_current_at|        snapshot_id|          parent_id|is_current_ancestor|
+--------------------+-------------------+-------------------+-------------------+
|2025-11-16 15:24:...|7504872170709576995|               NULL|               true|
|2025-11-16 15:24:...|8573225573759967105|7504872170709576995|               true|
|2025-11-16 15:25:...|3927684976588199350|8573225573759967105|               true|
+--------------------+-------------------+-------------------+-------------------+



In [ ]:
spark.stop()